# Exercise 03: Regression Trees, Random Forest and Gradient Boosting – Concrete Strength

**Dataset:** 1'030 concrete samples tested in the laboratory (Yeh, 1998). For each sample the mixture (kg per m³ of
concrete), the age at testing (days) and the measured compressive strength (MPa) are known.
Source: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength)
/ [Kaggle](https://www.kaggle.com/datasets/maajdl/yeh-concret-data).

| variable | meaning |
|---|---|
| `cement`, `slag`, `flyash`, `water`, `superplasticizer`, `coarseaggregate`, `fineaggregate` | components in kg/m³ |
| `age` | age of the sample at testing (days, 1–365) |
| `csMPa` | compressive strength (MPa) – **target** |

**Business question:** A construction company wants to predict the strength of a new concrete mixture *before* waiting
28 days for the laboratory test. Strength depends on the components in a strongly non-linear way and with interactions
(e.g. water-to-cement ratio, hardening over time) – a typical use case for tree-based models.

**Slides:** Part 03 «Regression models» – regression trees, pruning, hyperparameters, bagging, random forest, gradient
boosting, model comparison.

**After this exercise you can ...**
- find the best split of a regression tree by hand (SSE criterion),
- fit, visualise and prune a regression tree,
- explain and apply bagging, random forests (incl. out-of-bag error, permutation importance) and gradient boosting,
- compare models fairly with cross-validation.

## Concept

**Regression tree.** The tree cuts the feature space into boxes and predicts one constant per box: the mean of the
training targets in that box. A split (feature $j$, threshold $s$) is chosen greedily so that the total squared error of
the two children is minimal:

$$\min_{j,\,s}\; \Big[\sum_{x_i \in R_1}(y_i-\bar y_{R_1})^2 + \sum_{x_i \in R_2}(y_i-\bar y_{R_2})^2\Big]$$

**Cost-complexity pruning.** $R_\alpha(T) = \text{SSE}(T) + \alpha\,|T|$, where $|T|$ is the number of leaves. Larger
$\alpha$ → smaller tree. In scikit-learn: `ccp_alpha`, chosen by cross-validation.

**Why averaging helps.** For $B$ models with variance $\sigma^2$ and pairwise correlation $\rho$:

$$\operatorname{Var}\Big(\frac{1}{B}\sum_{b=1}^B \hat f_b\Big) = \rho\,\sigma^2 + \frac{1-\rho}{B}\,\sigma^2$$

- **Bagging:** fit deep trees on $B$ bootstrap samples and average. Each tree does not see ≈ 37 % of the observations
  (*out-of-bag*, OOB) → internal error estimate.
- **Random forest:** bagging + only $m$ of the $p$ features are considered at each split → de-correlated trees
  (smaller $\rho$) → lower variance.
- **Gradient boosting:** small trees are added one after the other; each new tree $h_m$ fits the current residuals
  (squared loss): $\hat f_m(x) = \hat f_{m-1}(x) + \nu\, h_m(x)$ with learning rate $\nu$.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: Calculations by hand

**a) Best split of a regression tree.** Six samples of the same mixture were tested at different ages:

| age $x$ [days] | 3 | 7 | 14 | 28 | 56 | 90 |
|---|---|---|---|---|---|---|
| strength $y$ [MPa] | 15 | 22 | 30 | 38 | 42 | 44 |

Compute the SSE of the root node. Then compute, for each of the five possible thresholds (midpoints between neighbouring
ages), the means and the SSE of the two child nodes. Which split does the tree choose? What are the leaf predictions?

**b) Variance of an average.** 100 trees have variance $\sigma^2 = 4$ each. Compute the variance of their average for
$\rho = 0.6$ (bagged trees, similar to each other) and for $\rho = 0.2$ (random forest). What happens for
$B \to \infty$?

**c) One boosting step.** The current prediction for a sample is 30 MPa, the observed strength is 40 MPa. The next tree
predicts the residual exactly. What is the new prediction with learning rate $\nu = 0.1$? How many such steps are needed
until the remaining residual is below 5 MPa?

**d)** Verify your results with Python.

In [ ]:
# a) Best split
x = np.array([3, 7, 14, 28, 56, 90])
y = np.array([15, 22, 30, 38, 42, 44])

# Your code here

# b) Variance of an average
# Your code here

# c) Boosting steps
# Your code here

**Your answer:** ...

## Exercise 2: Import and explore the data

a) Import the data, check for missing values and duplicates (remove duplicates), and show descriptive statistics.

b) Create scatter plots of `csMPa` against `cement`, `water` and `age`. Which relationships are clearly non-linear?

c) Split the data into training (80 %) and test set (20 %) with `random_state=42`.

In [ ]:
# a) Import the data
df = pd.read_csv('./Data/concrete_data.csv', sep=',')

# Your code here

# b) Scatter plots
# Your code here

# c) Train/test split
X = df.drop(columns='csMPa')
y = df['csMPa']
X_train, X_test, y_train, y_test = ___

**Your answer:** ...

## Exercise 3: Baseline – linear regression

Fit a multiple linear regression with all features on the training set and compute RMSE and $R^2$ on the test set. This
is the baseline that the tree-based models have to beat.

In [ ]:
# Your code here

## Exercise 4: A small regression tree

a) Fit a `DecisionTreeRegressor(max_depth=3, random_state=42)` on the training set.

b) Visualise the tree with `tree.plot_tree()` and interpret the first two levels. Which features are used?

c) Compute RMSE and $R^2$ on the test set. How does the small tree compare to the linear regression?

In [ ]:
# a) Fit the tree
reg_tree = ___

# b) Plot the tree
# Your code here

# c) Test RMSE and R2
# Your code here

**Your answer:** ...

## Exercise 5: Tree complexity and pruning

a) For `max_depth` = 1, 2, …, 20 compute the RMSE on the training set and the 5-fold CV RMSE (on the training set).
Plot both curves. Where does the tree start to overfit?

b) Alternatively, grow a full tree and prune it: compute the candidate values with
`DecisionTreeRegressor(random_state=42).cost_complexity_pruning_path(X_train, y_train).ccp_alphas` and choose `ccp_alpha`
with `GridSearchCV` (5-fold CV, `scoring='neg_root_mean_squared_error'`). How many leaves does the best pruned tree have?
What is its test RMSE?

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# a) Training vs. CV error for different depths
# Your code here

# b) Cost-complexity pruning
# Your code here

**Your answer:** ...

## Exercise 6: Random forest

a) Fit random forests with `n_estimators` = 1, 5, 10, 25, 50, 100, 200, 400 (`oob_score=True`, `random_state=42`,
`n_jobs=-1`). The OOB predictions are stored in `oob_prediction_`. Plot the OOB RMSE against the number of trees. When do
the gains plateau?

b) Compare `max_features` = 1.0 (all features, i.e. bagging), `'sqrt'` and 0.5 with 5-fold CV (300 trees).

c) For the best forest, compute the test RMSE. Compare the impurity-based feature importance (`feature_importances_`)
with the **permutation importance** on the test set.

In [ ]:
# a) Number of trees vs. OOB error
n_trees = [1, 5, 10, 25, 50, 100, 200, 400]
oob_rmse = []
for n in n_trees:
    # Your code here
    pass

# b) max_features
# Your code here

# c) Test RMSE and feature importance
# Your code here

In [ ]:
# c) Test RMSE and feature importance (use the best max_features from b)
# Your code here

**Your answer:** ...

## Exercise 7: Gradient boosting

a) Fit `GradientBoostingRegressor(n_estimators=1000, max_depth=3, random_state=42)` with learning rates
$\nu$ = 0.01, 0.1 and 1.0. Use `staged_predict(X_test)` to compute the test RMSE after each boosting iteration and plot
the three curves.

*Note: we look at the test curve here only for illustration. In practice, `learning_rate` and `n_estimators` are tuned
with cross-validation (see Exercise 8).*

b) Describe the influence of the learning rate. What happens with $\nu = 1.0$?

In [ ]:
# a) Test error per iteration for different learning rates
plt.figure(figsize=(8, 4))
for lr in [0.01, 0.1, 1.0]:
    gb = ___
    rmse_staged = [root_mean_squared_error(y_test, p) for p in gb.staged_predict(X_test)]
    plt.plot(range(1, 1001), rmse_staged, label=f'learning rate {lr}')
plt.ylim(0, 20)
plt.xlabel('Number of trees M')
plt.ylabel('Test RMSE [MPa]')
plt.legend()
plt.show()

**Your answer:** ...

## Exercise 8: Fair model comparison

Compare the following models with **5-fold CV RMSE on the training set** and summarise the results in a table:
linear regression, pruned tree (best `ccp_alpha` from Exercise 5), random forest (best setting from Exercise 6) and
gradient boosting (`learning_rate=0.1`, `n_estimators=500`, `max_depth=3`). Then evaluate the best model **once** on the
test set. Which model would you recommend to the construction company, and what are the drawbacks?

In [ ]:
models = {
    'Linear regression': LinearRegression(),
    'Pruned tree':       ___,
    'Random forest':     ___,
    'Gradient boosting': ___,
}

# Your code here

**Your answer:** ...

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')